In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from unidecode import unidecode
import yaml

def normalize_column_name(df):
    result = []
    for col in df.columns:
        col = re.sub('(\n|\t)', '', col) # replace \n and \t with ''
        col = col.lower() # lower
        col = re.sub(' ', '_', col) # repace ' ' with '_'
        col = unidecode(col) # remove accent

        result.append(col)

    df.columns = result

    return df

df_fact_selection = normalize_column_name(pd.read_csv(r'..\\data\fact_selection.csv'))
df_fact_current_round = normalize_column_name(pd.read_csv(r'..\data\fact_current_round.csv'))

df_dim_team = normalize_column_name(pd.read_csv(r'..\\data\dim_team.csv'))
df_dim_score = normalize_column_name(pd.read_csv(r'..\\data\dim_score.csv'))

def create_leaderboard(scenario):
    """
    Create the leaderboard based on scenario
    """

    
    df_fact_selection = normalize_column_name(pd.read_csv(r'..\\data\fact_selection.csv'))
    df_fact_current_round = normalize_column_name(pd.read_csv(r'..\data\fact_current_round.csv'))

    df_dim_team = normalize_column_name(pd.read_csv(r'..\\data\dim_team.csv'))
    df_dim_score = normalize_column_name(pd.read_csv(r'..\\data\dim_score.csv'))
    

    # change the df_fact_current_round based on the scenario:
    content = scenario['content']
    for team, current_round in content.items():
        df_fact_current_round.loc[df_fact_current_round['team'] == team, 'current_round'] = current_round

        if current_round == 'final':
            df_fact_current_round.loc[df_fact_current_round['team'] == team, 'team_status'] = 0
        elif current_round == 'winner':
            df_fact_current_round.loc[df_fact_current_round['team'] == team, 'team_status'] = 1

    # continue

    df_fact_current_score = df_dim_team.merge(
        df_fact_current_round
        ,left_on="team"
        ,right_on="team"
    ).merge(
        df_dim_score
        ,left_on=['pot', 'current_round']
        ,right_on=['pot', 'round']
    )

    df_fact_selection['new_team_id'] = (
        df_fact_selection.merge(
            df_fact_current_score
            ,left_on='team'
            ,right_on='team'
        )
        .sort_values(by=['ticket', 'pot', 'team'])
        .groupby(['ticket']).cumcount() + 1
    )

    df_fact_selection['team_id'] = 'team_' + df_fact_selection['new_team_id'].astype(str)
    df_fact_selection = (
        df_fact_selection.drop(['new_team_id'], axis=1)
    )

    df_fact_score = df_fact_selection.merge(
            df_fact_current_score
            ,left_on='team'
            ,right_on='team'
        ).assign(team = lambda x: x['team'] + ' (pot ' + x['pot'].astype(str) + ')')

    df_fact_score_agg =(
        df_fact_selection.merge(
            df_fact_current_score
            ,left_on='team'
            ,right_on='team'
        )
        .sort_values(by=['pot'])
        .groupby(['ticket'])
        .agg(
            total_score = ('score', 'sum')
            ,pot_selected = ('pot', lambda x: ''.join(x.astype(str)))
            ,pot_avg = ('pot', 'mean')
        )
        .reset_index()
        .sort_values(by='total_score', ascending=False)
        .reset_index(drop=True)
    )

    df_fact_score_agg['rank'] = df_fact_score_agg.index + 1

    def create_df_team():
        # Create an empty dictionary to hold all our DataFrames
        team_dfs = {}
        
        for i in [1, 2, 3, 4]:
            # Assign each new DataFrame to a dynamically named dictionary key
            team_dfs[f'df_team_{i}'] = (
                df_fact_score[df_fact_score['team_id'] == f'team_{i}']
                .rename(columns={
                    'team': f'team_{i}',
                    'current_round': f'round_{i}',
                    'score': f'score_{i}'
                })
                [['ticket', f'team_{i}', f'round_{i}', f'score_{i}']]
            )
            
        return team_dfs


    list_cols = ['rank','ticket', 'total_score', 'pot_selected']
    for i in [1,2,3,4]:
        list_cols += [f"team_{i}", f"round_{i}", f"score_{i}"]

    df_fact_score_agg = (
        df_fact_score_agg
        .merge(
            create_df_team()['df_team_1']
            ,left_on='ticket'
            ,right_on='ticket'
        )
        .merge(
                create_df_team()['df_team_2']
                ,left_on='ticket'
                ,right_on='ticket'
        )
        .merge(
            create_df_team()['df_team_3']
            ,left_on='ticket'
            ,right_on='ticket'
        )
        .merge(
            create_df_team()['df_team_4']
            ,left_on='ticket'
            ,right_on='ticket'
        )
    )[list_cols]

    return df_fact_score_agg

In [5]:
scenario_path = r"..\data\scenario.yaml"
with open(scenario_path, "r") as file:
    scenario_file = yaml.safe_load(file)

for i, scenario in enumerate(scenario_file['scenario']):
    print(scenario)

{'id': 'scenario_5', 'content': {'Spain': 'winner', 'Argentina': 'final'}, 'name': 'Chung kết = Spain, Argentina --- Vô địch: Spain'}
{'id': 'scenario_6', 'content': {'Spain': 'final', 'Argentina': 'winner'}, 'name': 'Chung kết = Spain, Argentina --- Vô địch: Argentina'}
{'id': 'scenario_7', 'content': {'Spain': 'winner', 'England': 'final'}, 'name': 'Chung kết = Spain, England --- Vô địch: Spain'}
{'id': 'scenario_8', 'content': {'Spain': 'final', 'England': 'winner'}, 'name': 'Chung kết = Spain, England --- Vô địch: England'}


In [6]:
dict_round_decode = {
        'group_stage': 1
        ,'32_team': 2
        ,'16_team': 3
        ,'quarter_final': 4
        ,'semi_final': 5
        ,'final': 6
        ,'winner': 7
    }

In [7]:
dim_user = normalize_column_name(pd.read_csv(r"..\data\standing.csv").dropna(how='all'))['ticket'].str.split(" - ").str[0].drop_duplicates().sort_values(key=lambda col: col.apply(unidecode)).tolist()

dict_user = {}

for user in dim_user:
    dict_user[user] = {}
    dict_user[user]['top_1_scenario'] = []
    dict_user[user]['top_3_scenario'] = []

    for active_scenario, scenario in enumerate(scenario_file['scenario']):
        df = create_leaderboard(scenario_file['scenario'][active_scenario])
        list_cols_round = [col for col in df.columns if 'round' in col and 'decode' not in col]
        list_cols_round_decode = []
        for col in list_cols_round:
            list_cols_round_decode.append(f'{col}_decoded')
            df[f'{col}_decoded'] = df[col].map(dict_round_decode)


        df['round_agg'] = (
            df[list_cols_round_decode]
            .astype(str).agg(''.join, axis=1)
            .apply(lambda row: int(''.join(sorted(list(row), reverse=True))))
        )


        df = (
            df
            .drop(list_cols_round_decode, axis=1)
            .sort_values(by=['total_score', 'round_agg'], ascending=[False, False])
        )

        df['rank'] = df[['total_score', 'round_agg']].apply(lambda row: tuple(row), axis=1).rank(method='min', ascending=False).astype(int)
        df = df.drop(['round_agg'], axis=1)



        if df.head(1)['ticket'].str.contains(user).sum() > 0:
            dict_user[user]['top_1_scenario'].append(scenario_file['scenario'][active_scenario]['name'])
            
        if df.head(3)['ticket'].str.contains(user).sum() > 0:
            dict_user[user]['top_3_scenario'].append(scenario_file['scenario'][active_scenario]['name'])

        dict_user[user]['num_top_1_scenario'] = len(dict_user[user]['top_1_scenario'])
        dict_user[user]['num_top_3_scenario'] = len(dict_user[user]['top_3_scenario'])

        

dict_user

{'Đặng Huy Hoàng': {'top_1_scenario': [],
  'top_3_scenario': [],
  'num_top_1_scenario': 0,
  'num_top_3_scenario': 0},
 'Đinh Việt Long': {'top_1_scenario': [],
  'top_3_scenario': [],
  'num_top_1_scenario': 0,
  'num_top_3_scenario': 0},
 'Đỗ Bá Thành': {'top_1_scenario': [],
  'top_3_scenario': [],
  'num_top_1_scenario': 0,
  'num_top_3_scenario': 0},
 'Huỳnh Ngọc Minh': {'top_1_scenario': ['Chung kết = Spain, Argentina --- Vô địch: Spain',
   'Chung kết = Spain, Argentina --- Vô địch: Argentina'],
  'top_3_scenario': ['Chung kết = Spain, Argentina --- Vô địch: Spain',
   'Chung kết = Spain, Argentina --- Vô địch: Argentina',
   'Chung kết = Spain, England --- Vô địch: Spain',
   'Chung kết = Spain, England --- Vô địch: England'],
  'num_top_1_scenario': 2,
  'num_top_3_scenario': 4},
 'Lê Thanh Tùng': {'top_1_scenario': [],
  'top_3_scenario': [],
  'num_top_1_scenario': 0,
  'num_top_3_scenario': 0},
 'Lương Hồ Nam': {'top_1_scenario': [],
  'top_3_scenario': [],
  'num_top_1_s

In [8]:
import pandas as pd
import streamlit as st

# 1. Create data where one cell contains a list
st.dataframe(
    pd.DataFrame(dict_user).T,
    column_config={
        "Players": st.column_config.TextColumn("top_1_scenario", width="large")
    },
    use_container_width=True
)


2026-07-15 08:06:09.506 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-07-15 08:06:09.568 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 08:06:10.504 
  command:

    streamlit run C:\Users\minhtl3\AppData\Roaming\Python\Python313\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-07-15 08:06:10.505 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 08:06:10.505 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

In [9]:
df = pd.DataFrame(dict_user).T
columns_to_unpack = ['top_1_scenario', 'top_3_scenario']
for col in columns_to_unpack:
    df[col] = df[col].apply(lambda x: ';'.join(x) if isinstance(x, list) else x)



df = (
    df
    .reset_index()
    .rename(
        {
            'index': 'User'
            ,'top_1_scenario': 'Kịch bản top 1'
            ,'top_3_scenario': 'Kịch bản top 3'
            ,'num_top_1_scenario': 'Số kịch bản top 1'
            ,'num_top_3_scenario': 'Số kịch bản top 3'
        }
        , axis=1
    )
    [[
        'User'
        ,'Số kịch bản top 1'
        ,'Kịch bản top 1'
        ,'Số kịch bản top 3'
        ,'Kịch bản top 3'
    ]]
)

df.to_csv('..\data\scenario_by_user.csv', index=False)


df.style.set_properties(**{'white-space': 'pre-wrap'})


# df = df.style.format({
#     'Số kịch bản top 1': hide_zeros,
#     'Số kịch bản top 3': hide_zeros
# })

# df

<>:30: SyntaxWarning: invalid escape sequence '\d'
<>:30: SyntaxWarning: invalid escape sequence '\d'
C:\Users\minhtl3\AppData\Local\Temp\ipykernel_18480\3529123312.py:30: SyntaxWarning: invalid escape sequence '\d'
  df.to_csv('..\data\scenario_by_user.csv', index=False)


,User,Số kịch bản top 1,Kịch bản top 1,Số kịch bản top 3,Kịch bản top 3
0,Đặng Huy Hoàng,0,,0,
1,Đinh Việt Long,0,,0,
2,Đỗ Bá Thành,0,,0,
3,Huỳnh Ngọc Minh,2,"Chung kết = Spain, Argentina --- Vô địch: Spain;Chung kết = Spain, Argentina --- Vô địch: Argentina",4,"Chung kết = Spain, Argentina --- Vô địch: Spain;Chung kết = Spain, Argentina --- Vô địch: Argentina;Chung kết = Spain, England --- Vô địch: Spain;Chung kết = Spain, England --- Vô địch: England"
4,Lê Thanh Tùng,0,,0,
5,Lương Hồ Nam,0,,0,
6,Mai Hoàng Nam,0,,0,
7,Nguyễn Duy Đăng,0,,0,
8,Nguyễn Hoàng Long,0,,0,
9,Nguyễn Hữu Long,0,,0,


In [10]:
df = (pd.read_csv(r"../data/scenario_by_user.csv").dropna(how='all'))


dim_user = df['User'].tolist()
dropdown_user = ['Tất cả'] + dim_user
dropdown_user

['Tất cả',
 'Đặng Huy Hoàng',
 'Đinh Việt Long',
 'Đỗ Bá Thành',
 'Huỳnh Ngọc Minh',
 'Lê Thanh Tùng',
 'Lương Hồ Nam',
 'Mai Hoàng Nam',
 'Nguyễn Duy Đăng',
 'Nguyễn Hoàng Long',
 'Nguyễn Hữu Long',
 'Nguyễn Quang Anh EB',
 'Nguyễn Văn Khang',
 'Nguyễn Việt Đức',
 'Phạm Minh Quân',
 'Phan Công Phi',
 'Phan Thị Việt Thương',
 'Tào Lê Minh',
 'Trần Thị Huyền Trang',
 'Trần Xuân Huy',
 'Vũ Thành Công']